# 00 — Data Loading & Sanity Checks

Load BOEM lease-sale files and the OCS block shapefile using `boem_loader`.

**Sales loaded:** 198 (available now), 257, 261, Dec 2025 (add data directories once downloaded).

**Outputs:** row counts, column heads, basic spatial plot.

In [ ]:
import sys, os

# Add mvp0/ to path so we can import the helper module
ROOT = os.path.abspath(os.path.join("..", ".."))
sys.path.insert(0, os.path.join(ROOT, "mvp0"))

import boem_loader as bl
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 1. Define sale directories

Each BOEM sale ZIP should be unzipped into its own folder under `data/lease-sales/`.

| Directory | Sale | Status |
|-----------|------|--------|
| `sale_198` | Sale 198 | available |
| `sale_257` | Sale 257 (Aug 2023) | download from BOEM |
| `sale_261` | Sale 261 (Mar 2024) | download from BOEM |
| `sale_dec2025` | OBBBA Dec 2025 | download from BOEM |

In [ ]:
DATA = os.path.join(ROOT, "data")
LEASE_DIR = os.path.join(DATA, "lease-sales")
SHP_PATH = os.path.join(DATA, "shapefiles", "blocks.shp")

SALE_DIRS = {
    198: os.path.join(LEASE_DIR, "sale_198"),
    257: os.path.join(LEASE_DIR, "sale_257"),
    261: os.path.join(LEASE_DIR, "sale_261"),
    "dec2025": os.path.join(LEASE_DIR, "sale_dec2025"),
}

# Show which sale directories exist
for label, path in SALE_DIRS.items():
    status = "OK" if os.path.isdir(path) else "MISSING"
    print(f"  Sale {label:>8}  {status:>7}  {path}")

## 2. Load Sale 198 (sample data)

In [ ]:
s198 = bl.load_sale(SALE_DIRS[198])

print("Loaded tables:", list(s198.keys()))
for name, df in s198.items():
    print(f"  {name:12s}  {df.shape[0]:>5} rows  x  {df.shape[1]} cols")

In [ ]:
print("=== Tracts (PREBID) ===")
s198["tracts"].head()

In [ ]:
print("=== Bids ===")
s198["bids"].head()

In [ ]:
print("=== Companies ===")
s198["companies"].head()

In [ ]:
print("=== Maps (protraction area lookup) ===")
if "maps" in s198:
    display(s198["maps"].head())
else:
    print("(not present in this sale)")

### Sanity checks — Sale 198

In [ ]:
tracts = s198["tracts"]
bids = s198["bids"]
companies = s198["companies"]

# 1. Lease-number join coverage
tract_leases = set(tracts["Lease_Number"])
bid_leases = set(bids["Lease_Number"])
print(f"Unique tract leases : {len(tract_leases)}")
print(f"Unique bid leases   : {len(bid_leases)}")
print(f"Bid leases in tracts: {len(bid_leases & tract_leases)} / {len(bid_leases)}")

# 2. Company coverage
bid_cos = set(bids["Company_Number"])
known_cos = set(companies["Company_Number"])
print(f"\nUnique bidding companies: {len(bid_cos)}")
print(f"Companies in lookup    : {len(known_cos)}")
print(f"Bid companies matched  : {len(bid_cos & known_cos)} / {len(bid_cos)}")
missing = bid_cos - known_cos
if missing:
    print(f"  Unmatched company codes: {missing}")

# 3. Quick bid stats
print(f"\nTotal bid rows      : {len(bids)}")
print(f"Total bid amount ($): {bids['Bid_Amount'].sum():,.0f}")
print(f"Mean bid ($)        : {bids['Bid_Amount'].mean():,.0f}")
print(f"Tracts with >=2 bids: {(tracts['Num_Bids'] >= 2).sum()} / {len(tracts)}")

## 3. Load additional sales (uncomment once data is downloaded)

Download each sale's ZIP from https://data.boem.gov/Main/Leasing.aspx,
unzip into the matching `data/lease-sales/sale_<id>/` folder, then run these cells.

In [ ]:
# --- Sale 257 (Aug 2023) ---
# s257 = bl.load_sale(SALE_DIRS[257])
# print("Sale 257:", {k: v.shape for k, v in s257.items()})

In [ ]:
# --- Sale 261 (Mar 2024) ---
# s261 = bl.load_sale(SALE_DIRS[261])
# print("Sale 261:", {k: v.shape for k, v in s261.items()})

In [ ]:
# --- December 2025 OBBBA Sale (validation only) ---
# s_dec25 = bl.load_sale(SALE_DIRS["dec2025"])
# print("Dec 2025:", {k: v.shape for k, v in s_dec25.items()})

## 4. Load OCS block shapefile

In [ ]:
blocks = bl.load_blocks(SHP_PATH, to_utm=True)

print(f"Blocks loaded : {len(blocks):,}")
print(f"CRS           : {blocks.crs}")
print(f"Columns       : {list(blocks.columns)}")
blocks.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
blocks.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.2)
ax.set_title("GOM OCS Blocks (UTM 15N)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

### Spatial join check — Sale 198 tracts to blocks

In [ ]:
# Join tracts to blocks via (Protraction_ID, Block_Number)
merged = tracts.merge(
    blocks[["Protraction_ID", "Block_Number", "geometry"]],
    on=["Protraction_ID", "Block_Number"],
    how="left",
)
matched = merged["geometry"].notna().sum()
print(f"Tracts matched to blocks: {matched} / {len(tracts)}")

if matched < len(tracts):
    unmatched = merged[merged["geometry"].isna()][
        ["Lease_Number", "Protraction_ID", "Block_Number"]
    ]
    print("Sample unmatched tracts:")
    display(unmatched.head(10))

## 5. Supplemental datasets (placeholders)

Download lease, well, and relinquishment CSVs from BOEM, place under `data/`, and use these loaders.

In [ ]:
# Example — uncomment and set paths once files are downloaded:
#
# leases = bl.load_leases(os.path.join(DATA, "leases.csv"))
# wells  = bl.load_wells(os.path.join(DATA, "wells.csv"))
# relin  = bl.load_relinquishments(os.path.join(DATA, "relinquishments.csv"))
#
# print(f"Leases: {leases.shape}  Wells: {wells.shape}  Relinquishments: {relin.shape}")

---

**Next:** Use these loaded DataFrames in the investigation notebooks:
- `01_adjacency_signal.ipynb` (Q1)
- `02_relinquishment_signal.ipynb` (Q2)
- `03_well_activity_signal.ipynb` (Q3)
- `04_archetype_stability.ipynb` (Q4)